In [1]:
from sentinelhub import DataCollection

for collection in DataCollection.get_available_collections():
    print(collection)

d:\Bussiness_plan\Multimodal_PM25\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DataCollection.SENTINEL2_L1C
DataCollection.SENTINEL2_L2A
DataCollection.SENTINEL1
DataCollection.SENTINEL1_IW
DataCollection.SENTINEL1_IW_ASC
DataCollection.SENTINEL1_IW_DES
DataCollection.SENTINEL1_EW
DataCollection.SENTINEL1_EW_ASC
DataCollection.SENTINEL1_EW_DES
DataCollection.SENTINEL1_EW_SH
DataCollection.SENTINEL1_EW_SH_ASC
DataCollection.SENTINEL1_EW_SH_DES
DataCollection.DEM
DataCollection.DEM_MAPZEN
DataCollection.DEM_COPERNICUS_30
DataCollection.DEM_COPERNICUS_90
DataCollection.MODIS
DataCollection.LANDSAT_MSS_L1
DataCollection.LANDSAT_TM_L1
DataCollection.LANDSAT_TM_L2
DataCollection.LANDSAT_ETM_L1
DataCollection.LANDSAT_ETM_L2
DataCollection.LANDSAT_OT_L1
DataCollection.LANDSAT_OT_L2
DataCollection.SENTINEL5P
DataCollection.SENTINEL3_OLCI
DataCollection.SENTINEL3_SLSTR
DataCollection.HARMONIZED_LANDSAT_SENTINEL


# A. Import + config

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timedelta

from sentinelhub import (
    CRS,
    BBox,
    DataCollection,
    MimeType,
    MosaickingOrder,
    SentinelHubRequest,
    SHConfig,
    bbox_to_dimensions,
)

# =========================
# CONFIG CDSE
# =========================
config = SHConfig()

config.sh_client_id = "sh-aba2f8e9-da97-47f3-8cae-a5a1fbeede2d"
config.sh_client_secret = "DPBslOq7z6pumggnCpQAQBDKfIE3fhQ9"

config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
config.sh_base_url = "https://sh.dataspace.copernicus.eu"


# B. Create CDSE collections

In [3]:
# Sentinel-2 L2A for CDSE
s2l2a_cdse = DataCollection.SENTINEL2_L2A.define_from(
    "S2L2A_CDSE",
    service_url=config.sh_base_url
)

# Sentinel-5P for CDSE
s5p_cdse = DataCollection.SENTINEL5P.define_from(
    "S5P_CDSE",
    service_url=config.sh_base_url
)

# C. Helper functions

In [4]:
def make_bbox(lat, lon, delta, crs=CRS.WGS84):
    return BBox([lon - delta, lat - delta, lon + delta, lat + delta], crs=crs)


def safe_stats(arr, prefix):
    arr = np.array(arr, dtype=np.float32)
    arr = arr[~np.isnan(arr)]
    
    if arr.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_median": np.nan,
        }
    
    return {
        f"{prefix}_mean": float(np.nanmean(arr)),
        f"{prefix}_std": float(np.nanstd(arr)),
        f"{prefix}_min": float(np.nanmin(arr)),
        f"{prefix}_max": float(np.nanmax(arr)),
        f"{prefix}_median": float(np.nanmedian(arr)),
    }

4) Sentinel-5P Feature Extractor
D. Sentinel-5P feature extractor

In [5]:
def get_s5p_single_band_features(lat, lon, date_str, band_name, delta=0.3, resolution=5000):
    """
    Fetch features for one Sentinel-5P band for one day.
    band_name examples: 'NO2', 'CO', 'SO2', 'AER_AI_340_380'
    """
    bbox = make_bbox(lat, lon, delta)
    size = bbox_to_dimensions(bbox, resolution=resolution)

    evalscript = f"""
    //VERSION=3
    function setup() {{
        return {{
            input: [{{
                bands: ["{band_name}", "dataMask"]
            }}],
            output: {{
                bands: 2,
                sampleType: "FLOAT32"
            }}
        }};
    }}

    function evaluatePixel(sample) {{
        return [sample.{band_name}, sample.dataMask];
    }}
    """

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=s5p_cdse,
                time_interval=(date_str, date_str),
                mosaicking_order=MosaickingOrder.MOST_RECENT
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],
        bbox=bbox,
        size=size,
        config=config
    )

    try:
        img = request.get_data()[0]
    except Exception as e:
        print(f"S5P {band_name} error on {date_str}: {e}")
        return {
            f"{band_name.lower()}_valid_pixels": np.nan,
            **safe_stats([], band_name.lower())
        }

    band = img[..., 0]
    mask = img[..., 1]

    band_masked = np.where(mask == 1, band, np.nan)

    features = {
        f"{band_name.lower()}_valid_pixels": int(np.sum(~np.isnan(band_masked)))
    }
    features.update(safe_stats(band_masked, band_name.lower()))
    return features

In [6]:
def get_s5p_features(lat, lon, date_str, delta=0.3, resolution=5000):
    """
    Fetch each Sentinel-5P product type separately and merge the results.
    """
    features = {"date": date_str}

    s5p_bands = ["NO2", "CO", "SO2", "AER_AI_340_380"]

    for band in s5p_bands:
        band_features = get_s5p_single_band_features(
            lat=lat,
            lon=lon,
            date_str=date_str,
            band_name=band,
            delta=delta,
            resolution=resolution
        )
        features.update(band_features)

    return features

In [7]:
# Configure parameters
lat, lon = 10.7769, 106.7009  # HCM_1 station coordinates
date_str = "2023-01-01"  # Specific date
band_name = "NO2"  # Or "CO", "SO2", "AER_AI_340_380"
features = get_s5p_single_band_features(lat, lon, date_str, band_name)
print(features)

{'no2_valid_pixels': 40, 'no2_mean': 3.8606798625551164e-05, 'no2_std': 1.2905560652143322e-05, 'no2_min': 1.894799788715318e-05, 'no2_max': 8.31860670587048e-05, 'no2_median': 3.664133691927418e-05}


5) Sentinel-2 Feature Extractor
E. Sentinel-2 feature extractor

In [8]:
def get_s2_features(lat, lon, date_str, delta=0.01, resolution=10):
    """
    Fetch Sentinel-2 L2A features for one day.
    date_str: 'YYYY-MM-DD'
    """
    bbox = make_bbox(lat, lon, delta)
    size = bbox_to_dimensions(bbox, resolution=resolution)

    evalscript = """
    //VERSION=3
    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04", "B08", "B11", "dataMask"],
                units: "REFLECTANCE"
            }],
            output: {
                bands: 6,
                sampleType: "FLOAT32"
            }
        };
    }

    function evaluatePixel(sample) {
        return [
            sample.B02,
            sample.B03,
            sample.B04,
            sample.B08,
            sample.B11,
            sample.dataMask
        ];
    }
    """

    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=s2l2a_cdse,
                time_interval=(date_str, date_str),
                mosaicking_order=MosaickingOrder.LEAST_CC
            )
        ],
        responses=[
            SentinelHubRequest.output_response("default", MimeType.TIFF)
        ],
        bbox=bbox,
        size=size,
        config=config
    )

    try:
        img = request.get_data()[0]
    except Exception as e:
        print(f"S2 error on {date_str}: {e}")
        return {
            "date": date_str,
            "s2_valid_pixels": np.nan,
            **safe_stats([], "ndvi"),
            **safe_stats([], "ndbi"),
            **safe_stats([], "ndwi"),
        }

    b2 = img[..., 0]
    b3 = img[..., 1]
    b4 = img[..., 2]
    b8 = img[..., 3]
    b11 = img[..., 4]
    mask = img[..., 5]

    b2 = np.where(mask == 1, b2, np.nan)
    b3 = np.where(mask == 1, b3, np.nan)
    b4 = np.where(mask == 1, b4, np.nan)
    b8 = np.where(mask == 1, b8, np.nan)
    b11 = np.where(mask == 1, b11, np.nan)

    # Indices
    ndvi = (b8 - b4) / (b8 + b4 + 1e-6)
    ndbi = (b11 - b8) / (b11 + b8 + 1e-6)
    ndwi = (b3 - b8) / (b3 + b8 + 1e-6)

    features = {
        "date": date_str,
        "s2_valid_pixels": int(np.sum(~np.isnan(ndvi)))
    }
    features.update(safe_stats(ndvi, "ndvi"))
    features.update(safe_stats(ndbi, "ndbi"))
    features.update(safe_stats(ndwi, "ndwi"))

    return features

6) Build Dataset by Date
F. Create the Date List

In [9]:
def generate_date_list(start_date, end_date):
    dates = pd.date_range(start_date, end_date, freq="D")
    return [d.strftime("%Y-%m-%d") for d in dates]

G. Build full feature table

In [10]:
def build_satellite_feature_dataset(lat, lon, start_date, end_date):
    date_list = generate_date_list(start_date, end_date)

    rows = []
    for d in date_list:
        print(f"Processing {d} ...")
        
        s5p_feat = get_s5p_features(lat, lon, d)
        s2_feat = get_s2_features(lat, lon, d)

        row = {"date": d, "lat": lat, "lon": lon}
        row.update({k: v for k, v in s5p_feat.items() if k != "date"})
        row.update({k: v for k, v in s2_feat.items() if k != "date"})

        dt = pd.to_datetime(d)
        row["day_of_week"] = dt.dayofweek
        row["month"] = dt.month
        row["day_of_year"] = dt.dayofyear

        rows.append(row)

    return pd.DataFrame(rows)

7) Example Satellite Dataset Run

In [11]:
lat = 10.78533
lon = 106.67029

df_sat = build_satellite_feature_dataset(
    lat=lat,
    lon=lon,
    start_date="2024-01-01",
    end_date="2024-01-10"
)

df_sat.head()

Processing 2024-01-01 ...
Processing 2024-01-02 ...
Processing 2024-01-03 ...
Processing 2024-01-04 ...
Processing 2024-01-05 ...
Processing 2024-01-06 ...
Processing 2024-01-07 ...
Processing 2024-01-08 ...
Processing 2024-01-09 ...
Processing 2024-01-10 ...


,date,lat,lon,no2_valid_pixels,no2_mean,no2_std,no2_min,no2_max,no2_median,co_valid_pixels,...,ndbi_max,ndbi_median,ndwi_mean,ndwi_std,ndwi_min,ndwi_max,ndwi_median,day_of_week,month,day_of_year
0,2024-01-01,10.78533,106.67029,0,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,1
1,2024-01-02,10.78533,106.67029,126,0.000045,0.000017,0.000019,0.000103,0.000040,158,...,0.535936,0.072390,-0.063614,0.089372,-0.534271,0.224160,-0.048677,1,1,2
2,2024-01-03,10.78533,106.67029,98,0.000043,0.000015,0.000018,0.000104,0.000040,163,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,1,3
3,2024-01-04,10.78533,106.67029,113,0.000047,0.000022,0.000011,0.000116,0.000041,151,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,1,4
4,2024-01-05,10.78533,106.67029,1,0.000040,0.000000,0.000040,0.000040,0.000040,65,...,-0.015672,-0.176534,0.157046,0.011476,0.131411,0.205298,0.154505,4,1,5


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# INPUT FILES
# =========================
FILE_2161292 = Path(r"D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi/2161292_LuuQuangVu_pm25_features_strict.csv")
FILE_2161306 = Path(r"D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi/2161306_MinhKhai_pm25_features_strict.csv")

OUTPUT_DIR = Path("satellite_two_stations_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOAD 2 STATIONS
# =========================
df1 = pd.read_csv(FILE_2161292)
df2 = pd.read_csv(FILE_2161306)

df_hourly = pd.concat([df1, df2], ignore_index=True)

df_hourly["datetime_utc"] = pd.to_datetime(df_hourly["datetime_utc"], utc=True, errors="coerce")
df_hourly = df_hourly.dropna(subset=["datetime_utc"]).copy()
df_hourly["date"] = df_hourly["datetime_utc"].dt.strftime("%Y-%m-%d")

print("Hourly combined shape:", df_hourly.shape)
display(df_hourly.head())

# =========================
# DAILY PM2.5 FOR SATELLITE MERGE
# =========================
df_pm25_daily = (
    df_hourly
    .groupby(
        ["location_id", "location_name", "latitude", "longitude", "date"],
        as_index=False
    )
    .agg(
        pm25=("pm25", "mean"),
        pm25_daily_std=("pm25", "std"),
        pm25_hour_count=("pm25", "count")
    )
)

print("Daily PM2.5 shape:", df_pm25_daily.shape)
display(df_pm25_daily.head())

# =========================
# BUILD SATELLITE FEATURES ONLY FOR DATES THAT EXIST
# =========================
def build_satellite_feature_dataset_for_station_dates(location_id, location_name, lat, lon, date_list):
    rows = []

    for d in sorted(date_list):
        print(f"[{location_id}] Processing {d} ...")

        row = {
            "location_id": location_id,
            "location_name": location_name,
            "latitude": lat,
            "longitude": lon,
            "date": d,
        }

        # Sentinel-5P
        try:
            s5p_feat = get_s5p_features(lat, lon, d)
            row.update({k: v for k, v in s5p_feat.items() if k != "date"})
        except Exception as e:
            print(f"  S5P failed on {d}: {e}")

        # Sentinel-2
        try:
            s2_feat = get_s2_features(lat, lon, d)
            row.update({k: v for k, v in s2_feat.items() if k != "date"})
        except Exception as e:
            print(f"  S2 failed on {d}: {e}")

        dt = pd.to_datetime(d)
        row["day_of_week"] = dt.dayofweek
        row["month"] = dt.month
        row["day_of_year"] = dt.dayofyear

        rows.append(row)

    return pd.DataFrame(rows)

# =========================
# UNIQUE STATIONS
# =========================
stations_meta = (
    df_pm25_daily[["location_id", "location_name", "latitude", "longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(stations_meta)

# =========================
# RUN SATELLITE FOR EACH STATION
# =========================
satellite_all = []
merged_all = []

for row in stations_meta.itertuples(index=False):
    location_id = row.location_id
    location_name = row.location_name
    lat = row.latitude
    lon = row.longitude

    station_dates = (
        df_pm25_daily.loc[df_pm25_daily["location_id"] == location_id, "date"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print(f"\n===== BUILD SATELLITE FOR {location_id} | {location_name} =====")
    print(f"Num dates: {len(station_dates)}")

    df_sat_station = build_satellite_feature_dataset_for_station_dates(
        location_id=location_id,
        location_name=location_name,
        lat=lat,
        lon=lon,
        date_list=station_dates
    )

    # save satellite-only per station
    safe_name = str(location_name).replace("/", "_").replace("\\", "_").replace(",", "").strip()
    sat_path = OUTPUT_DIR / f"{location_id}_{safe_name}_satellite_daily.csv"
    df_sat_station.to_csv(sat_path, index=False, encoding="utf-8-sig")
    print("Saved satellite-only:", sat_path.resolve())

    # merge with PM2.5 daily
    df_merged_station = df_sat_station.merge(
        df_pm25_daily,
        on=["location_id", "location_name", "latitude", "longitude", "date"],
        how="left"
    )

    merged_path = OUTPUT_DIR / f"{location_id}_{safe_name}_satellite_pm25_daily.csv"
    df_merged_station.to_csv(merged_path, index=False, encoding="utf-8-sig")
    print("Saved merged station:", merged_path.resolve())
    print("Merged shape:", df_merged_station.shape)

    satellite_all.append(df_sat_station)
    merged_all.append(df_merged_station)

# =========================
# COMBINED FILES
# =========================
df_sat_all = pd.concat(satellite_all, ignore_index=True).sort_values(["location_id", "date"])
df_merged_all = pd.concat(merged_all, ignore_index=True).sort_values(["location_id", "date"])

sat_all_path = OUTPUT_DIR / "all_stations_satellite_daily.csv"
merged_all_path = OUTPUT_DIR / "all_stations_satellite_pm25_daily.csv"

df_sat_all.to_csv(sat_all_path, index=False, encoding="utf-8-sig")
df_merged_all.to_csv(merged_all_path, index=False, encoding="utf-8-sig")

print("\n=== DONE ===")
print("Satellite all:", sat_all_path.resolve(), df_sat_all.shape)
print("Merged all   :", merged_all_path.resolve(), df_merged_all.shape)

display(df_merged_all.head(10))

Hourly combined shape: (16208, 21)


,datetime_utc,location_id,location_name,latitude,longitude,pm25,year,month,day,hour,...,is_weekend,is_dry_season,pm25_lag1,pm25_lag24,pm25_lag168,pm25_rolling_3h,pm25_rolling_24h,pm25_rolling_7d,pm25_rolling_24h_std,date
0,2024-02-07 10:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.99,2024,2,7,10,...,0,1,10.54,12.05,48.63,6.29,5.427500,18.521250,2.273485,2024-02-07
1,2024-02-07 11:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.62,2024,2,7,11,...,0,1,17.99,8.88,52.74,11.02,5.675000,18.338869,3.171679,2024-02-07
2,2024-02-07 12:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.03,2024,2,7,12,...,0,1,13.62,6.73,49.70,14.05,5.872500,18.106012,3.509518,2024-02-07
3,2024-02-07 13:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,20.06,2024,2,7,13,...,0,1,13.03,6.33,53.02,14.88,6.135000,17.887738,3.800030,2024-02-07
4,2024-02-07 14:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,24.13,2024,2,7,14,...,0,1,20.06,5.93,49.56,15.57,6.707083,17.691548,4.746340,2024-02-07


Daily PM2.5 shape: (804, 8)


,location_id,location_name,latitude,longitude,date,pm25,pm25_daily_std,pm25_hour_count
0,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-07,12.780714,6.415772,14
1,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-08,5.235417,2.052216,24
2,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-09,15.841250,12.576299,24
3,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-10,15.087917,6.051613,24
4,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-11,12.793750,4.480779,24


,location_id,location_name,latitude,longitude
0,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999
1,2161306,Minh Khai - Bắc Từ Liêm,21.0500,105.7400



===== BUILD SATELLITE FOR 2161292 | Số 46, phố Lưu Quang Vũ =====
Num dates: 579
[2161292] Processing 2024-02-07 ...
[2161292] Processing 2024-02-08 ...
[2161292] Processing 2024-02-09 ...
[2161292] Processing 2024-02-10 ...
[2161292] Processing 2024-02-11 ...
[2161292] Processing 2024-02-12 ...
[2161292] Processing 2024-02-13 ...
[2161292] Processing 2024-02-14 ...
[2161292] Processing 2024-02-15 ...
[2161292] Processing 2024-02-16 ...
[2161292] Processing 2024-02-17 ...
[2161292] Processing 2024-02-18 ...
[2161292] Processing 2024-02-19 ...
[2161292] Processing 2024-02-20 ...
[2161292] Processing 2024-02-21 ...
[2161292] Processing 2024-02-22 ...
[2161292] Processing 2024-02-23 ...
[2161292] Processing 2024-02-24 ...
[2161292] Processing 2024-02-25 ...
[2161292] Processing 2024-02-26 ...
[2161292] Processing 2024-02-27 ...
[2161292] Processing 2024-02-28 ...
[2161292] Processing 2024-02-29 ...
[2161292] Processing 2024-03-01 ...
[2161292] Processing 2024-03-02 ...
[2161292] Processi

,location_id,location_name,latitude,longitude,date,no2_valid_pixels,no2_mean,no2_std,no2_min,no2_max,...,ndwi_std,ndwi_min,ndwi_max,ndwi_median,day_of_week,month,day_of_year,pm25,pm25_daily_std,pm25_hour_count
0,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-07,0,NaN,NaN,NaN,NaN,...,0.011806,-0.014812,0.083573,0.011765,2,2,38,12.780714,6.415772,14
1,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-08,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3,2,39,5.235417,2.052216,24
2,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-09,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4,2,40,15.841250,12.576299,24
3,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-10,105,0.000028,0.000008,0.000011,0.000055,...,NaN,NaN,NaN,NaN,5,2,41,15.087917,6.051613,24
4,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-11,155,0.000028,0.000008,0.000009,0.000046,...,NaN,NaN,NaN,NaN,6,2,42,12.793750,4.480779,24
5,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-12,156,0.000063,0.000032,0.000023,0.000166,...,0.122985,-0.929708,0.302856,-0.236555,0,2,43,13.363333,4.884199,24
6,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-13,39,0.000058,0.000018,0.000029,0.000088,...,NaN,NaN,NaN,NaN,1,2,44,8.086667,2.697317,24
7,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-14,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2,2,45,7.585000,2.348805,24
8,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-15,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3,2,46,9.715833,3.585751,24
9,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,2024-02-16,12,0.000076,0.000029,0.000039,0.000132,...,NaN,NaN,NaN,NaN,4,2,47,7.277500,2.579557,24
